# 08 — Sensitivity / Scenario Analysis

**Owner:** Person B (decision track).

**Rubric line:** Sensitivity / simulation analysis. **This is what
separates a 4 from a 5** — show that the recommendation holds across
plausible perturbations of the assumptions.


In [ ]:
# --- Setup --------------------------------------------------------------
# Make `src/` importable regardless of where the notebook is launched from.
import sys, pathlib
PROJECT_ROOT = pathlib.Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config, data, features, models, metrics, decision, viz

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


## 8.1 — Load test predictions

In [ ]:
import joblib
_, test_df = data.load_interim()
y_test = test_df[config.TARGET_COL].values
lgbm_artifact = joblib.load(config.MODELS_DIR / 'improved_lgbm.joblib')
y_proba = lgbm_artifact['proba_test']


## 8.2 — Scenario 1: cost / value perturbations

Sweep cost-per-contact and value-per-conversion across plausible ranges.
Reports EV and decision threshold at each cell.


In [ ]:
sweep = decision.sensitivity_sweep(
    y_test, y_proba,
    cost_per_contact_grid=[2.5, 5.0, 10.0, 20.0],
    value_per_conversion_grid=[60, 120, 200, 300],
)
sweep.to_csv(config.TABLES_DIR / 'sensitivity_cost_value.csv', index=False)
sweep


## 8.3 — Scenario 2: contact budget perturbations

In [ ]:
rows = []
for frac in [0.05, 0.10, 0.20, 0.30, 0.50]:
    rule = decision.build_decision_rule(y_test, y_proba, contact_budget_fraction=frac)
    rows.append({
        'budget_fraction': frac,
        'threshold': rule.threshold,
        'expected_value_eur': rule.expected_value_eur,
        'contacts': rule.contacts,
        'successful': rule.successful,
        'wasted': rule.wasted,
        'missed': rule.missed,
    })
budget_sweep = pd.DataFrame(rows)
budget_sweep.to_csv(config.TABLES_DIR / 'sensitivity_budget.csv', index=False)
budget_sweep


## 8.4 — Scenario 3: recession (base rate halves)

Simulate a downturn by downsampling positives in the test set so the
base rate is half its current value, then re-evaluate the rule.


In [ ]:
rng = np.random.default_rng(config.RANDOM_SEED)
test_idx = np.arange(len(y_test))
pos_idx = test_idx[y_test == 1]
neg_idx = test_idx[y_test == 0]
keep_pos = rng.choice(pos_idx, size=len(pos_idx) // 2, replace=False)
recession_idx = np.sort(np.concatenate([keep_pos, neg_idx]))
rec_rule = decision.build_decision_rule(y_test[recession_idx], y_proba[recession_idx])
print(rec_rule.describe())


## 8.5 — What holds, what shifts

Three or four bullets summarising what is robust (the *direction* of
the recommendation) and what is sensitive (which specific threshold,
which expected value).
